In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import numpy as np
import pandas as pd

# Create pipeline dictionary with 5 models and class_weight='balanced'
models = {
    'Logistic Regression': Pipeline([
        ('model', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42))
    ]),
    'KNN': Pipeline([
        ('model', KNeighborsClassifier())
    ]),
    'SVM': Pipeline([
        ('model', SVC(class_weight='balanced', probability=True, random_state=42))
    ]),
    'Decision Tree': Pipeline([
        ('model', DecisionTreeClassifier(class_weight='balanced', random_state=42))
    ]),
    'Random Forest': Pipeline([
        ('model', RandomForestClassifier(class_weight='balanced', random_state=42, n_estimators=200))
    ])
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []

for name, model in models.items():
    # Cross-validation on training data
    cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='f1', n_jobs=None)

    # Fit on training data and predict on test data
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_test, y_proba) if y_proba is not None else np.nan

    results.append({
        'Model': name,
        'CV_F1_Mean': cv_scores.mean(),
        'CV_F1_Std': cv_scores.std(),
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1_Score': f1,
        'ROC_AUC': roc_auc
    })

model_comparison = pd.DataFrame(results).sort_values(by='F1_Score', ascending=False)
print(model_comparison)
print("\nBest Model based on F1 Score:")
best_model_name = model_comparison.iloc[0]['Model'] 
print(best_model_name)  

<VSCode.Cell id="#VSC-b9ad9cac" language="python">
